# In-season minutes model

Predict fixture-level minutes from lagged player performance at several
forecast horizons. Models are trained and evaluated with a chronological
gameweek split.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from catboost import CatBoostRegressor, EFeaturesSelectionAlgorithm, Pool
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from prediction.artifacts.io import MINUTES_ARTIFACT_PATH, save_trained_catboost_model
from training.config import MAX_HORIZON, NUMERIC_FEATURES, RANDOM_STATE, ROLLING_WINDOWS
from training.load_training_data import load_historic_player_fixture_data
from training.transformations import create_fixture_horizon_df, create_trended_calculations

TRAINING_SEASONS = ["2022-23", "2023-24"]
TARGET_COLUMN = "minutes"
VALIDATION_FRACTION = 0.20
MAX_FEATURES = 20
CATEGORICAL_COLUMNS = ["position", "target_gw", "current_gw", "horizon"]
CATBOOST_PARAMS = {
    "iterations": 163,
    "learning_rate": 0.03,
    "depth": 6,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

## Load data

In [ ]:
fixture_history_df = pd.concat(
    [
        load_historic_player_fixture_data(season)
        .rename(columns={"GW": "target_gw"})
        .assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
).sort_values(["player_id", "season", "target_gw", "fixture_id"])

print(fixture_history_df.groupby("season").size())

## Engineer features

In [ ]:
calculated_features_df = create_trended_calculations(
    fixture_history_df,
    rolling_windows=ROLLING_WINDOWS,
    numeric_features=NUMERIC_FEATURES,
)
fixture_history_df = pd.concat(
    [fixture_history_df.reset_index(drop=True), calculated_features_df.reset_index(drop=True)],
    axis=1,
)
calculated_columns = calculated_features_df.columns.tolist()
model_features = CATEGORICAL_COLUMNS + calculated_columns

## Create forecast horizons

Each target fixture is expanded to one row per forecast horizon. `current_gw`
is the latest gameweek whose results are available, and
`horizon = target_gw - current_gw`.

In [ ]:
fixture_horizon_df = create_fixture_horizon_df(
    fixture_history_df=fixture_history_df,
    max_horizon=MAX_HORIZON,
    calculated_columns=calculated_columns,
    target_column=TARGET_COLUMN,
)

## Split training and validation data

In [ ]:
# Keep every horizon for a fixture on the same side of the chronological split.
fixture_gameweeks = (
    fixture_horizon_df[["season", "target_gw"]]
    .drop_duplicates()
    .sort_values(["season", "target_gw"])
)
validation_count = max(1, int(np.ceil(len(fixture_gameweeks) * VALIDATION_FRACTION)))
validation_fixture_gameweeks = pd.MultiIndex.from_frame(
    fixture_gameweeks.tail(validation_count)
)
row_fixture_gameweeks = pd.MultiIndex.from_frame(
    fixture_horizon_df[["season", "target_gw"]]
)
validation_mask = row_fixture_gameweeks.isin(validation_fixture_gameweeks)

X_train = fixture_horizon_df.loc[~validation_mask, model_features].copy()
X_valid = fixture_horizon_df.loc[validation_mask, model_features].copy()
y_train = fixture_horizon_df.loc[~validation_mask, TARGET_COLUMN].copy()
y_valid = fixture_horizon_df.loc[validation_mask, TARGET_COLUMN].copy()

for frame in (X_train, X_valid):
    frame[CATEGORICAL_COLUMNS] = (
        frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    )

print(f"Training rows: {len(X_train):,}; validation rows: {len(X_valid):,}")

## Train candidate models

In [ ]:
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train[["horizon"]], y_train)
validation_predictions["Dummy"] = dummy_model.predict(X_valid[["horizon"]])

full_model = CatBoostRegressor(**CATBOOST_PARAMS).fit(
    X_train, y_train, cat_features=CATEGORICAL_COLUMNS
)
validation_predictions["CatBoost (all features)"] = full_model.predict(X_valid)

In [ ]:
# Keep every categorical feature and select only from the numeric features.
numeric_feature_count = min(
    max(0, MAX_FEATURES - len(CATEGORICAL_COLUMNS)), len(calculated_columns)
)
if numeric_feature_count == len(calculated_columns):
    selected_calculated_features = calculated_columns.copy()
else:
    selector = CatBoostRegressor(**CATBOOST_PARAMS)
    train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
    valid_pool = Pool(X_valid, y_valid, cat_features=CATEGORICAL_COLUMNS)
    selection = selector.select_features(
        train_pool,
        eval_set=valid_pool,
        features_for_select=calculated_columns,
        num_features_to_select=numeric_feature_count,
        steps=min(10, len(calculated_columns) - numeric_feature_count),
        algorithm=EFeaturesSelectionAlgorithm.RecursiveByLossFunctionChange,
        train_final_model=False,
        verbose=False,
    )
    selected_names = set(selection["selected_features_names"])
    selected_calculated_features = [
        column for column in calculated_columns if column in selected_names
    ]

selected_features = CATEGORICAL_COLUMNS + selected_calculated_features
selected_categorical_columns = CATEGORICAL_COLUMNS.copy()
selected_model = CatBoostRegressor(**CATBOOST_PARAMS).fit(
    X_train[selected_features],
    y_train,
    cat_features=selected_categorical_columns,
)
validation_predictions["CatBoost (selected features)"] = selected_model.predict(
    X_valid[selected_features]
)
print(f"Selected features ({len(selected_features)}): {selected_features}")

## Evaluate models

In [ ]:
validation_context = fixture_horizon_df.loc[
    y_valid.index, ["season", "target_gw", "horizon"]
].reset_index(drop=True)

def evaluate_predictions(predictions):
    evaluation = validation_context.assign(
        actual=y_valid.to_numpy(), predicted=predictions
    )
    appearance_correct = evaluation["predicted"].gt(0).eq(
        evaluation["actual"].gt(0)
    )
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "appearance_hit_rate": appearance_correct.mean(),
        **{
            f"absolute_error_p{percentile}": evaluation["actual"]
            .sub(evaluation["predicted"])
            .abs()
            .quantile(percentile / 100)
            for percentile in (25, 50, 75, 90)
        },
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {
            name: evaluate_predictions(predictions)
            for name, predictions in validation_predictions.items()
        },
        orient="index",
    )
    .rename_axis("model")
    .reset_index()
    .sort_values(["appearance_hit_rate", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
comparison_table.round(3)

### Performance by forecast horizon

In [ ]:
horizon_records = []
for model_name, predictions in validation_predictions.items():
    evaluation = validation_context.assign(
        actual=y_valid.to_numpy(), predicted=predictions
    )
    for horizon, results in evaluation.groupby("horizon"):
        horizon_records.append({
            "model": model_name,
            "horizon": horizon,
            "appearance_hit_rate": results["predicted"].gt(0).eq(
                results["actual"].gt(0)
            ).mean(),
        })

horizon_results = pd.DataFrame(horizon_records)
figure, axis = plt.subplots(figsize=(12, 6))
for model_name, model_results in horizon_results.groupby("model"):
    axis.plot(
        model_results["horizon"],
        model_results["appearance_hit_rate"],
        marker="o",
        label=model_name,
    )
axis.set(
    xlabel="Forecast horizon (gameweeks)",
    ylabel="Appearance hit rate",
    title="Minutes > 0 appearance hit rate by forecast horizon",
    xticks=range(1, MAX_HORIZON + 1),
    ylim=(0, 1),
)
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()

## Explain selected model

In [ ]:
shap_sample_df = X_valid[selected_features].sample(
    n=min(10_000, len(X_valid)), random_state=RANDOM_STATE
)
shap_values = shap.TreeExplainer(selected_model)(shap_sample_df)
shap.plots.beeswarm(shap_values, max_display=len(selected_features), show=False)
plt.title("SHAP values for the in-season minutes model")
plt.tight_layout()
plt.show()

## Export selected model

In [ ]:
save_trained_catboost_model(
    model=selected_model,
    feature_columns=selected_features,
    categorical_columns=selected_categorical_columns,
    model_name="Unweighted CatBoost minutes model - top 20 features",
    model_version="0.1.0",
    save_path=MINUTES_ARTIFACT_PATH,
)
print(f"Saved minutes model to {MINUTES_ARTIFACT_PATH}")